# 用 scikit-learn 做一條 RAG 流程

大型語言模型 large language model（LLM）只知道訓練時看過的資料。要它回答**你手上文件**的內容，最直接的做法是：
先從文件裡找出和問題最相關的幾段，把這幾段連同問題一起交給模型，要求它只根據這些段落回答。
這個做法叫**檢索增強生成** retrieval-augmented generation（RAG）。

這本筆記本用 9 篇介紹亞洲國家的英文短文（其中一篇是台灣），從頭做一條 RAG 流程，分四步：

| 步 | 做什麼 | 用到的工具 |
|---|---|---|
| 1 切塊 | 把文件切成一小段一小段，叫做 chunk | Python 字串 |
| 2 嵌入 | 每段轉成一個 384 維的向量；意思相近的句子，向量方向也相近 | sentence-transformers 的 all-MiniLM-L6-v2 |
| 3 檢索 | 問題也轉成向量，找出方向最接近的 3 段 | scikit-learn 的 NearestNeighbors |
| 4 生成 | 3 段加上問題交給小型 LLM 回答 | Qwen2.5-0.5B-Instruct |

正式的系統常用向量資料庫，但 RAG 的核心只是「算向量之間的距離」。文件不多的時候，scikit-learn 就做得到，而且每一步的數字都看得到。
這裡用的模型全都公開，不用付費 API，也不用登入。

**怎麼執行**

1. 想快一點就先換 GPU：「執行階段 → 變更執行階段類型 → T4 GPU」。不換也能跑，只是每次生成答案要等比較久。
2. 由上往下逐格執行。第一次會從 Hugging Face 下載模型，生成模型約 1 GB，要等一兩分鐘。
3. 出現 `HF_TOKEN` 的提示可以不理：這些模型都是公開的，不登入也能下載。

## 0. 載入套件

Colab 已經內建這些套件，不用另外安裝。

In [ ]:
import glob
import os
import urllib.request
import zipfile

import numpy as np
import sentence_transformers
import sklearn
import torch
import transformers
from sentence_transformers import SentenceTransformer
from sklearn.neighbors import NearestNeighbors
from transformers import pipeline

# transformers 5 每次生成都會印幾行參數寫法的提醒，不影響結果，關掉讓輸出好讀
transformers.logging.set_verbosity_error()

print('scikit-learn', sklearn.__version__, '｜sentence-transformers', sentence_transformers.__version__,
      '｜transformers', transformers.__version__, '｜torch', torch.__version__)
print('GPU：', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '沒有，用 CPU 執行')

## 1. 下載文件

9 篇短文打包成一個約 5 KB 的壓縮檔，放在公開的 GitHub repo。每篇是一個 `.txt` 檔，60 到 80 個英文字。

In [ ]:
URL = 'https://github.com/gakudo-ai/open-datasets/blob/main/asia_documents.zip?raw=true'
ZIP_PATH = 'asia_documents.zip'
DOC_DIR = 'asia_documents'

urllib.request.urlretrieve(URL, ZIP_PATH)
with zipfile.ZipFile(ZIP_PATH) as z:
    z.extractall(DOC_DIR)

documents, doc_names = [], []
# 【Colab 版修改】加 sorted()：glob 的順序在不同系統不一樣，排序後切塊編號每次相同
for path in sorted(glob.glob(f'{DOC_DIR}/**/*.txt', recursive=True)):
    with open(path, encoding='utf-8') as f:
        documents.append(f.read())
    doc_names.append(os.path.basename(path))

print(f'讀到 {len(documents)} 篇文件')
for name, doc in zip(doc_names, documents):
    print(f'  {name:<16} {len(doc.split())} 個字')
print()
print('台灣那一篇的全文：')
print(documents[doc_names.index('Taiwan.txt')])

## 2. 切塊

一篇文件可能很長，整篇轉成一個向量，細節會被平均掉；切成小段，檢索才找得到「剛好講到這件事」的那一段。

這裡每塊 25 個字，相鄰兩塊重疊 5 個字。重疊是為了不讓一句話剛好被切在兩塊中間、兩邊都看不懂。
實際系統常用幾百字一塊；這裡刻意切小，9 篇短文才切得出效果。

In [ ]:
def chunk_text(text, chunk_size=25, overlap=5):
    """把文字切成每塊 chunk_size 個字、相鄰兩塊重疊 overlap 個字。"""
    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size - overlap):
        chunks.append(' '.join(words[i:i + chunk_size]))
    return chunks


all_chunks, chunk_source = [], []
for name, doc in zip(doc_names, documents):
    pieces = chunk_text(doc)
    all_chunks.extend(pieces)
    chunk_source.extend([name] * len(pieces))  # 記下每塊來自哪一篇，回答時才標得出出處

print(f'{len(documents)} 篇文件切成 {len(all_chunks)} 塊')
print()
first = doc_names[0]
print(f'以 {first} 為例，切成 {chunk_source.count(first)} 塊。注意每塊開頭的 5 個字，就是上一塊最後的 5 個字：')
for i in range(chunk_source.count(first)):
    print(f'  [{i}] {all_chunks[i]}')

## 3. 嵌入：把每一塊變成向量

all-MiniLM-L6-v2 是一個專門把句子轉成向量的小模型，約 2,300 萬個參數，輸出 384 維。
它訓練的目標是：意思相近的兩句話，向量的方向要相近。

In [ ]:
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
chunk_embeddings = embedding_model.encode(all_chunks)

print('向量矩陣的形狀：', chunk_embeddings.shape, '（塊數, 維度）')
print('第 0 塊的前 5 個數字：', np.round(chunk_embeddings[0][:5], 4))

## 4. 檢索：找出最接近問題的幾塊

用 scikit-learn 的 `NearestNeighbors` 當作向量資料庫：

- `metric='cosine'`：比的是兩個向量的**夾角**，不管長度。距離 = 1 − cos θ，方向完全相同是 0，越不相關越大。
- `algorithm='brute'`：問題和每一塊都算一次距離。31 塊一下子就算完；文件很多時才需要更快的索引。

In [ ]:
class SklearnRetriever:
    def __init__(self, n_neighbors=3):
        self.retriever = NearestNeighbors(n_neighbors=n_neighbors, metric='cosine', algorithm='brute')
        self.chunks = []

    def fit(self, chunks, embeddings):
        """建立索引。"""
        self.chunks = chunks
        self.retriever.fit(embeddings)

    def retrieve(self, query_embedding):
        """回傳最接近的幾塊：[(段落文字, 餘弦距離, 塊編號), ...]，距離由小到大。"""
        distances, indices = self.retriever.kneighbors([query_embedding])
        return [(self.chunks[i], d, i) for d, i in zip(distances[0], indices[0])]


vector_db = SklearnRetriever(n_neighbors=3)
vector_db.fit(all_chunks, chunk_embeddings)
print(f'索引建好了：{len(all_chunks)} 塊，每次找最接近的 3 塊')

先不接 LLM，單獨看檢索找到什麼：

In [ ]:
q = 'Which Asian countries are best known for islands and beaches?'
print('問題：', q)
for text, dist, idx in vector_db.retrieve(embedding_model.encode(q)):
    print(f'  距離 {dist:.3f}｜{chunk_source[idx]:<16}｜{text[:70]}...')

## 5. 生成：把找到的段落交給 LLM

Qwen2.5-0.5B-Instruct 約 5 億個參數，經過指令微調，會照對話格式回答。它夠小，Colab 免費版不會記憶體不足。

交給模型的內容分兩部分：

- **system**：規定它只能根據提供的段落回答，找不到就說找不到。
- **user**：檢索到的段落，接著是問題。

`do_sample=False` 表示每一步都選機率最高的字，同一個問題每次答案都一樣。

In [ ]:
# 【Colab 版修改】精度固定 float32：transformers 5 預設改用模型自帶的 bfloat16，T4 GPU 沒有 bfloat16 硬體支援
llm = pipeline('text-generation', model='Qwen/Qwen2.5-0.5B-Instruct', device_map='auto', dtype=torch.float32)

SYSTEM = ("You are a helpful assistant. Answer the user's question using ONLY the provided context. "
          "If the answer is not in the context, say 'I cannot find the answer in the provided documents.'")


def generate(messages, show_prompt=False):
    """把對話套上模型的格式後生成，只回傳新生成的文字。"""
    prompt = llm.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    if show_prompt:
        print('===== 模型實際收到的文字 =====')
        print(prompt)
        print('==============================')
    # 【Colab 版修改】return_full_text=False 只回傳新生成的部分，不再用字串長度切掉 prompt；
    # 拿掉 temperature：do_sample=False 時它不起作用
    out = llm(prompt, max_new_tokens=150, do_sample=False, return_full_text=False)
    return out[0]['generated_text'].strip()


def generate_rag_answer(query, retrieved_context, show_prompt=False):
    context_str = ' --- '.join(retrieved_context)
    messages = [
        {'role': 'system', 'content': SYSTEM},
        {'role': 'user', 'content': f'Context:\n{context_str}\n\nQuestion: {query}'},
    ]
    return generate(messages, show_prompt)

## 6. 串起來：問一個問題

三步：問題轉成向量 → 找最接近的 3 塊 → 交給 LLM 回答。第一次執行時順便印出「模型實際收到的文字」，看看檢索到的段落是怎麼塞進去的。

In [ ]:
def ask_rag_pipeline(question, show_prompt=False):
    print(f'問題：{question}')
    query_vector = embedding_model.encode(question)                    # 1. 問題轉成向量
    hits = vector_db.retrieve(query_vector)                             # 2. 找最接近的 3 塊
    answer = generate_rag_answer(question, [h[0] for h in hits], show_prompt)  # 3. 生成
    print(f'\nRAG 回答：{answer}')
    print('\n--- 用到的段落 ---')
    for k, (text, dist, idx) in enumerate(hits, 1):
        print(f'{k}. 距離 {dist:.3f}｜{chunk_source[idx]}｜{text[:90]}...')


ask_rag_pipeline('Which Asian countries are best known for islands and beaches?', show_prompt=True)

## 7. 對照：不檢索、直接問

同一個模型、同一個問題，一次直接問，一次先檢索再問。比一比：哪一個答案裡的內容，在文件裡真的找得到？

In [ ]:
def ask_without_rag(question):
    return generate([{'role': 'user', 'content': question}])


q = "What are Taiwan's most famous natural attractions?"
print('【不檢索，直接問】')
print(ask_without_rag(q))
print()
print('【先檢索再問】')
ask_rag_pipeline(q)

## 8. 問一個文件裡沒有的事

9 篇短文都沒有提到人口。照 system 的規定，模型應該回答找不到。

實際跑的結果：這個 0.5B 的小模型沒有守規矩，照樣說出一個人口數字，而且下面列出的段落裡根本沒有這個數字。
這就是 RAG 要附上「用到的段落」的原因：答案要能對回出處，對不回去的就不能信。

In [ ]:
ask_rag_pipeline('What is the population of Japan?')

## 9. 延伸：用中文問

all-MiniLM-L6-v2 只學過英文。用中文問台灣的景點，看看它找到什麼、距離多少：

In [ ]:
q_zh = '台灣最有名的自然景點是什麼？'
print('英文嵌入模型，問題：', q_zh)
for text, dist, idx in vector_db.retrieve(embedding_model.encode(q_zh)):
    print(f'  距離 {dist:.3f}｜{chunk_source[idx]:<16}｜{text[:60]}...')

換成多語嵌入模型 paraphrase-multilingual-MiniLM-L12-v2：它學過 50 多種語言，同一個意思不管用哪種語言說，向量都會靠在一起。
輸出一樣是 384 維，只要換模型、重新算一次段落向量，其他程式都不用改。第一次執行要下載約 470 MB。

生成模型 Qwen 本來就會中文，在 system 加一句「用提問的語言回答」即可。比一比兩個模型找到的段落與距離。

Qwen 回答中文時常用簡體字。實測要求它「用繁體中文回答」，這個小模型反而開始編造答案，所以這裡不加這個要求。

In [ ]:
multi_model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
multi_db = SklearnRetriever(n_neighbors=3)
multi_db.fit(all_chunks, multi_model.encode(all_chunks))

print('多語嵌入模型，問題：', q_zh)
hits = multi_db.retrieve(multi_model.encode(q_zh))
for text, dist, idx in hits:
    print(f'  距離 {dist:.3f}｜{chunk_source[idx]:<16}｜{text[:60]}...')

messages = [
    {'role': 'system', 'content': SYSTEM + ' Answer in the same language as the question.'},
    {'role': 'user', 'content': 'Context:\n' + ' --- '.join(h[0] for h in hits) + f'\n\nQuestion: {q_zh}'},
]
print()
print('RAG 回答：', generate(messages))

## 10. 自己試試

1. 把第 6 段的問題換成別的，例如問蒙古的節慶、越南的地理景點；第 9 段也換一個中文問題，例如「蒙古有什麼傳統節慶？」。
2. 第 4 段的 `n_neighbors` 改成 1 或 5，重跑第 4 到 6 段：給模型的段落變少或變多，答案有什麼不同？
3. 第 2 段的 `chunk_size` 改成 50、`overlap` 改成 0，重跑第 2 到 6 段：塊數變多少？答案有沒有變？
4. 把第 1 段的 `documents` 換成你自己的英文文字，例如研究計畫摘要（`documents = ['...', '...']`，`doc_names` 也給同樣數量的名稱），再從第 2 段往下跑。

**和正式系統差在哪**

- 文件多到幾萬段以上時，逐一比對太慢，改用 FAISS 這類近似最近鄰索引，或專門的向量資料庫。
- 切塊通常依句子或段落切，每塊幾百字，不是固定字數。
- 嵌入與生成模型都會用大得多的版本。0.5B 的小模型容易答錯或不照指示，這裡用它是為了在免費 Colab 上跑得動。

---

本筆記本改編自 Iván Palomares Carrascosa〈Building a RAG Pipeline with LLM Embeddings and Scikit-learn〉，MachineLearningMastery.com，2026 年 10 月 13 日。
9 篇短文取自 gakudo-ai/open-datasets 的 `asia_documents.zip`，執行時從原處下載。